# بناء تطبيقات توليد الصور (OpenAI)

هناك أكثر من مجرد توليد النصوص في نماذج اللغة الكبيرة. يمكنك أيضًا توليد صور من أوصاف نصية. الصور كوسيط مفيد في مجالات الطب، العمارة، السياحة، تطوير الألعاب، التسويق، وأكثر. في هذا الدرس نعمل مع نماذج **GPT Image** الحالية على منصة OpenAI.

## أهداف التعلم

بنهاية هذا الدرس ستكون قادرًا على:

- شرح ما هو توليد الصور وأين يكون مفيدًا.
- فهم عائلة نماذج `gpt-image` وكيف تختلف عن نماذج DALL·E القديمة.
- بناء تطبيق توليد الصور وتحرير الصور.

## ما هو توليد الصور؟

تُنشئ نماذج توليد الصور صورًا بناءً على وصف نصي. تتعلم النماذج الحديثة مثل `gpt-image` العلاقة بين النص والصور أثناء التدريب، ثم تحول الضوضاء العشوائية تدريجيًا إلى صورة تطابق وصفك.

هناك عائلتان معروفتان من نماذج الصور:

- **`gpt-image` (OpenAI)** — الجيل الحالي المستخدم في هذا الدرس. يدعم التوليد من النص إلى الصورة وتحرير الصور (التلوين داخل قناع).
- **Midjourney** — نموذج تابع لطرف ثالث شهير له خدمته الخاصة وسير عمل عبر Discord.

> نماذج الصور القديمة من OpenAI — **DALL·E 2** و **DALL·E 3** — هي نماذج قديمة. لم يعد DALL·E 3 متاحًا للنشر الجديد، وميزات مثل `create_variation` كانت موجودة فقط في DALL·E 2. استخدم نماذج `gpt-image` للتطبيقات الجديدة.

> **مهم:** ترجع نماذج `gpt-image` الصورة المولدة كـ **base64** (`b64_json`)، وليس كعنوان URL. يقوم كودك بفك شفرة سلسلة base64 إلى بايتات ويحفظها — لا يوجد عنوان URL للصورة لتنزيلها.


## بناء تطبيق إنشاء صورك الأول

فما الذي تحتاجه لبناء تطبيق إنشاء صور؟ أنت بحاجة إلى المكتبات التالية:

- **python-dotenv**، يُنصح بشدة باستخدام هذه المكتبة لحفظ أسرارك في ملف *.env* بعيدًا عن الكود.
- **openai**، هذه المكتبة هي ما ستستخدمه للتفاعل مع واجهة برمجة تطبيقات OpenAI.
- **pillow**، للعمل مع الصور في بايثون.


1. أنشئ ملفًا باسم *.env* بالمحتوى التالي:

    ```text
    OPENAI_API_KEY='<add your OpenAI key here>'
    ```



1. اجمع المكتبات المذكورة أعلاه في ملف يسمى *requirements.txt* كما يلي:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. بعد ذلك، قم بإنشاء بيئة افتراضية وتثبيت المكتبات:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!ملاحظة]
> لنظام ويندوز، استخدم الأوامر التالية لإنشاء بيئتك الافتراضية وتنشيطها:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. أضف الكود التالي في ملف يسمى *app.py*:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai

    # استيراد dotenv
    dotenv.load_dotenv()

    # إنشاء كائن OpenAI (يقرأ OPENAI_API_KEY من ملف .env الخاص بك)
    client = openai.OpenAI()


    try:
        # إنشاء صورة باستخدام واجهة برمجة تطبيقات توليد الصور
        generation_response = client.images.generate(
            model="gpt-image-1",
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # أدخل نص المطالبة الخاص بك هنا
            size='1024x1024',
            n=1
        )
        # تعيين الدليل للصورة المخزنة
        image_dir = os.path.join(os.curdir, 'images')

        # إذا لم يكن الدليل موجودًا، فأنشئه
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # تهيئة مسار الصورة (لاحظ أن نوع الملف يجب أن يكون png)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # نماذج gpt-image تعيد الصورة بتنسيق base64 (b64_json)، وليس كعنوان URL
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # عرض الصورة في العارض الافتراضي للصور
        image = Image.open(image_path)
        image.show()

    # التقاط الاستثناءات
    except openai.BadRequestError as err:
        print(err)

    ```

لنشرح هذا الكود:

- أولاً، نستورد المكتبات التي نحتاجها، بما في ذلك مكتبة OpenAI، مكتبة dotenv، وحدة base64، ومكتبة Pillow.

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai
    ```

- بعد ذلك، ننشئ العميل، الذي يقرأ مفتاح API من ملف ``.env`` الخاص بك.

    ```python
    # إنشاء كائن OpenAI
    client = openai.OpenAI()
    ```

- بعد ذلك، نولد الصورة:

    ```python
    generation_response = client.images.generate(
        model="gpt-image-1",
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # أدخل نص الموجه هنا
        size='1024x1024',
        n=1
    )
    ```

    نماذج `gpt-image` تُرجع الصورة كسلسلة **base64** في `data[0].b64_json`. نحن نفك تشفيرها إلى بايتات ونكتبها إلى ملف — لا يوجد رابط URL للتحميل.

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- أخيراً، نفتح الصورة ونستخدم عارض الصور الافتراضي لعرضها:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### مزيد من التفاصيل حول توليد الصورة

دعونا نلقي نظرة على معلمات `images.generate`:

- **model**، هو نموذج الصورة الذي سيتم استخدامه (على سبيل المثال `gpt-image-1`).
- **prompt**، هو النص المستخدم لتوليد الصورة. هنا هو "أرنب على حصان، يحمل مصاصة، في مرج ضبابي حيث تنمو زهور النرجس".
- **size**، هو حجم الصورة المولدة (على سبيل المثال `1024x1024`، `1536x1024`، `1024x1536`، أو `"auto"`).
- **n**، هو عدد الصور المولدة. هنا نولد صورة واحدة.

> نماذج الصور لا تأخذ معلمة `temperature` — هذه خاصّة بالتحكم في توليد النصوص. للحصول على تنوع، استدعِ API مرة أخرى؛ ولتقليل التنوع، اجعل طلبك أكثر تحديداً.

## قدرات إضافية في توليد الصور

لقد رأيت كيف تولد صورة ببضع أسطر من بايثون. نماذج `gpt-image` يمكنها أيضاً **تحرير** صورة موجودة. من خلال توفير صورة، **قناع** اختياري (يحدد المنطقة التي سيتم تغييرها)، ونص، يمكنك تعديل جزء من الصورة — مثلاً، إضافة قبعة لأرنبنا.

```python
response = client.images.edit(
  model="gpt-image-1",
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# يتم أيضًا إرجاع التعديلات بتنسيق base64
edited_image = base64.b64decode(response.data[0].b64_json)
```

الصورة الأساسية تحتوي فقط على الأرنب؛ الصورة النهائية تحتوي على القبعة على الأرنب.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**تنويه**:
تمت ترجمة هذا المستند باستخدام خدمة الترجمة بالذكاء الاصطناعي [Co-op Translator](https://github.com/Azure/co-op-translator). بينما نسعى للدقة، يرجى العلم أن الترجمات الآلية قد تحتوي على أخطاء أو عدم دقة. يجب اعتبار المستند الأصلي بلغته الأصلية المصدر الرسمي والمعتمد. للمعلومات الهامة، يُنصح بالاستعانة بترجمة بشرية محترفة. نحن غير مسؤولين عن أي سوء فهم أو تفسير ناتج عن استخدام هذه الترجمة.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
